
<font color="purple" size="6"><b>SaaS Revenue & Churn Analysis</b></font>
<br>

This notebook performs analytical queries on a SaaS revenue dataset after the data cleaning and modeling stages.

The goal is to extract business insights for leadership using SQL queries executed on a SQLite analytical database.

The following analyses are included:

1. Rolling 12-month ARR per account
2. Month-over-month churn rate by product tier
3. Cohort analysis of renewal behavior
4. Days Sales Outstanding (DSO)
5. Downgrade detection using self-joins
6. Upsell detection using multi-level CTEs

<font color="purple" size="5"><b>Data Sources</b></font>
<br>

The analysis uses the following datasets:

- Accounts
- Usage Events
- Support Tickets
- Invoices
- Renewals
- NPS Surveys

All datasets were cleaned and standardized using a Python data pipeline before loading into a SQLite analytical database.

<font color="purple" size="4"><b>Import Libraries</b></font>

In [1]:
# Import Libraries

import pandas as pd
import sqlite3

<font color="purple" size="4"><b>Connecting to the Analytical Database</b></font>

The cleaned datasets were loaded into a SQLite database using a star schema structure.

The database contains:

Dimension Tables
- accounts_dim
- usage_dim
- support_dim
- nps_dim

Fact Tables
- renewals_fact
- invoices_fact

In [2]:
# connect to database
conn = sqlite3.connect("../Data/data_cleaned.db")


<font color="purple" size="4"><b>Verify Database Tables</b></font>


In [3]:
# test query
tables = pd.read_sql(
"""
SELECT name 
FROM sqlite_master 
WHERE type='table'
""",
conn)

tables

,name
0,accounts_dim
1,usage_dim
2,support_dim
3,nps_dim
4,renewals_fact
5,invoices_fact


<font color="purple" size="4"><b>Rolling 12-Month ARR per Account</b></font>

Annual Recurring Revenue (ARR) is a critical metric in SaaS companies.

To analyze revenue trends per account, we calculate a rolling 12-month ARR using SQL window functions.

This helps identify:

- Revenue growth
- Expansion opportunities
- Early churn signals

In [16]:
query = """
SELECT
    account_id,
    renewal_date,
    new_arr,
    SUM(new_arr) OVER (
        PARTITION BY account_id
        ORDER BY renewal_date
        ROWS BETWEEN 11 PRECEDING AND CURRENT ROW
    ) AS rolling_12m_arr
FROM renewals_fact
ORDER BY account_id, renewal_date;
"""

rolling_arr = pd.read_sql(query, conn)

rolling_arr.head()

,account_id,renewal_date,new_arr,rolling_12m_arr
0,1,2025-01-01,15860.00,15860.00
1,2,2025-01-01,20191.00,20191.00
2,3,2025-01-01,2718.29,2718.29
3,4,2025-01-01,0.00,0.00
4,5,2025-01-01,29423.00,29423.00


<font color="purple" size="4"><b>Month-over-Month Churn Rate by Product Tier</b></font>

Customer churn is one of the most important SaaS metrics.

In this analysis we measure:

Monthly churn rate segmented by product tier.

This helps leadership identify which customer segments are most at risk.

In [17]:
query = """
SELECT
    a.tier,
    strftime('%Y-%m', r.renewal_date) AS month,
    COUNT(*) AS total_accounts,
    SUM(CASE WHEN r.outcome = 'churned' THEN 1 ELSE 0 END) AS churned_accounts,
    ROUND(
        1.0 * SUM(CASE WHEN r.outcome = 'churned' THEN 1 ELSE 0 END) / COUNT(*),
        4
    ) AS churn_rate
FROM renewals_fact r
JOIN accounts_dim a
ON r.account_id = a.account_id
GROUP BY a.tier, month
ORDER BY month;
"""

churn = pd.read_sql(query, conn)

churn.head()

,tier,month,total_accounts,churned_accounts,churn_rate
0,Business,2025-01,128,19,0.1484
1,Enterprise,2025-01,127,17,0.1339
2,Growth,2025-01,128,14,0.1094
3,Starter,2025-01,122,24,0.1967


<font color="purple" size="4"><b>Cohort Analysis</b></font>

Cohort analysis helps evaluate retention performance based on the time customers joined.

We group accounts by their contract start quarter and track renewal behavior over time.

In [18]:
query = """
SELECT
    strftime('%Y', contract_start) || '-Q' ||
    ((CAST(strftime('%m', contract_start) AS INTEGER)-1)/3 +1) AS cohort_quarter,
    COUNT(account_id) AS accounts
FROM accounts_dim
GROUP BY cohort_quarter
ORDER BY cohort_quarter;
"""

cohort = pd.read_sql(query, conn)

cohort

,cohort_quarter,accounts
0,2024-Q2,119
1,2024-Q3,138
2,2024-Q4,124
3,2025-Q1,112
4,2025-Q2,7


<font color="purple" size="4"><b>Days Sales Outstanding (DSO)</b></font>

DSO measures the average number of days required to collect payment after issuing an invoice.

Lower DSO indicates efficient cash flow management.

In [19]:
query = """
SELECT
    a.region,
    AVG(julianday(i.paid_date) - julianday(i.invoice_date)) AS avg_dso
FROM invoices_fact i
JOIN accounts_dim a
ON i.account_id = a.account_id
WHERE i.paid_date IS NOT NULL
GROUP BY a.region;
"""

dso = pd.read_sql(query, conn)

dso

,region,avg_dso
0,APAC,37.048611
1,Europe,37.187387
2,North America,36.800813


<font color="purple" size="4"><b>Self Join - Downgrade within 60 days of ticket spike </b></font>

This analysis identifies accounts whose ARR decreased compared to the previous renewal period.

A downgrade may indicate declining product usage or dissatisfaction.

In [22]:
query = """
WITH ticket_spikes AS (
    SELECT
        account_id,
        DATE(created_at) AS ticket_date,
        COUNT(*) AS tickets
    FROM support_dim
    GROUP BY account_id, ticket_date
    HAVING COUNT(*) >= 3
)

SELECT
    r1.account_id,
    r1.renewal_date,
    r1.new_arr AS current_arr,
    r2.new_arr AS previous_arr,
    t.ticket_date
FROM renewals_fact r1
JOIN renewals_fact r2
ON r1.account_id = r2.account_id
AND r2.renewal_date = (
    SELECT MAX(renewal_date)
    FROM renewals_fact
    WHERE account_id = r1.account_id
    AND renewal_date < r1.renewal_date
)
JOIN ticket_spikes t
ON r1.account_id = t.account_id
WHERE r1.new_arr < r2.new_arr
AND ABS(julianday(r1.renewal_date) - julianday(t.ticket_date)) <= 60;
"""
downgrades = pd.read_sql(query, conn)
downgrades.head()

,account_id,renewal_date,current_arr,previous_arr,ticket_date


<font color="purple" size="4"><b>Multi-Level CTE Upsell Detection</b></font>

Upsells occur when the ARR of an account increases compared to the previous renewal.

Accounts with multiple upsells are strong expansion opportunities.

In [21]:
query = """
WITH arr_history AS (
    SELECT
        account_id,
        renewal_date,
        new_arr,
        LAG(new_arr) OVER (
            PARTITION BY account_id
            ORDER BY renewal_date
        ) AS previous_arr
    FROM renewals_fact
),

upsells AS (
    SELECT
        account_id,
        renewal_date
    FROM arr_history
    WHERE new_arr > previous_arr
)

SELECT
    account_id,
    COUNT(*) AS upsell_count
FROM upsells
GROUP BY account_id
HAVING COUNT(*) > 1;
"""

upsells = pd.read_sql(query, conn)

upsells.head()

,account_id,upsell_count


No accounts showed multiple upsell events in the available dataset.

This is likely due to the synthetic nature of the data where each account only has a single renewal record.

In [30]:
query = """
WITH upsells AS (
    SELECT
        account_id,
        renewal_date,
        new_arr,
        LAG(new_arr) OVER(
            PARTITION BY account_id
            ORDER BY renewal_date
        ) AS previous_arr
    FROM renewals_fact
)

SELECT *
FROM upsells
WHERE new_arr > previous_arr;
"""
top_renewals = pd.read_sql(query, conn)
top_renewals.head()

,account_id,renewal_date,new_arr,previous_arr


<font color="purple" size="4"><b>Key Business Insights</b></font>

The analysis highlights several strategic insights:

• Revenue growth trends across accounts  
• Product tiers with elevated churn risk  
• Customer cohorts with stronger retention  
• Regions with slower payment cycles  
• Accounts showing early downgrade signals  
• Accounts with strong expansion potential

These insights can help leadership improve retention strategies and optimize revenue growth.